In [1]:
import xarray as xr
import numpy as np
import dask
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [21]:
cluster = PBSCluster(
    cores=1,
    memory='64GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='1:00:00'
)
cluster.scale(jobs=4)
client = Client(cluster)
client

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39677 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/39677/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/39677/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.97:45303,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/39677/status,Total threads: 0
Started: Just now,Total memory: 0 B


2026-07-07 11:03:43,183 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyterhub.hpc.ucar.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the 

In [22]:
path='/glade/work/acruz/Caribbean_Heat_data/ERA5/'

In [23]:
MSDRSWRF = xr.open_zarr(path+'MSDRSWRF')
MSDRSWRF

<xarray.Dataset> Size: 16GB
Dimensions:                (forecast_initial_time: 33052, forecast_hour: 12,
                            latitude: 82, longitude: 121)
Coordinates:
  * forecast_initial_time  (forecast_initial_time) datetime64[ns] 264kB 1981-...
  * forecast_hour          (forecast_hour) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
Data variables:
    MSDRSWRF               (forecast_initial_time, forecast_hour, latitude, longitude) float32 16GB dask.array<chunksize=(8263, 12, 82, 121), meta=np.ndarray>

In [24]:
# joing forecast_i_time and forecast hour into single dim 'time'
MSDRSWRF_stack = MSDRSWRF.stack(time=('forecast_initial_time', 'forecast_hour'))
MSDRSWRF_stack

<xarray.Dataset> Size: 16GB
Dimensions:                (latitude: 82, longitude: 121, time: 396624)
Coordinates:
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
  * time                   (time) object 3MB MultiIndex
  * forecast_initial_time  (time) datetime64[ns] 3MB 1981-01-01T06:00:00 ... ...
  * forecast_hour          (time) int32 2MB 1 2 3 4 5 6 7 8 ... 6 7 8 9 10 11 12
Data variables:
    MSDRSWRF               (latitude, longitude, time) float32 16GB dask.array<chunksize=(82, 121, 99156), meta=np.ndarray>

In [ ]:
actual_time = MSDRSWRF_stack['forecast_initial_time'] + MSDRSWRF_stack['forecast_hour'].astype('timedelta64[h]')

In [26]:
MSDRSWRF_fxtime = MSDRSWRF_stack.assign_coords(fixed_time=actual_time)
MSDRSWRF_fxtime = MSDRSWRF_fxtime.sortby('time').drop_duplicates('time')
MSDRSWRF_fxtime

<xarray.Dataset> Size: 16GB
Dimensions:                (latitude: 82, longitude: 121, time: 396624)
Coordinates:
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
  * time                   (time) object 3MB MultiIndex
  * forecast_initial_time  (time) datetime64[ns] 3MB 1981-01-01T06:00:00 ... ...
  * forecast_hour          (time) int32 2MB 1 2 3 4 5 6 7 8 ... 6 7 8 9 10 11 12
    fixed_time             (time) datetime64[ns] 3MB 1981-01-01T07:00:00 ... ...
Data variables:
    MSDRSWRF               (latitude, longitude, time) float32 16GB dask.array<chunksize=(82, 121, 99156), meta=np.ndarray>

In [27]:
MSDRSWRF_clim = MSDRSWRF_fxtime.groupby('fixed_time.month').mean()
MSDRSWRF_clim

<xarray.Dataset> Size: 478kB
Dimensions:    (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    MSDRSWRF   (month, latitude, longitude) float32 476kB dask.array<chunksize=(12, 82, 121), meta=np.ndarray>

In [ ]:
MSDRSWRF_clim.to_netcdf(path+'MSDRSWRF_clim.nc')

In [20]:
client.shutdown()